# Versie 2

## Algemene info
In het gegeven mechanisme zijn er 3 "beweegbare" loops, en 1 loop waarvan we de positie K wensen te kennen.
Hieronder staan de vergelijkingen van de loops:

Loop 1: $\vec{AB} + \vec{BD} -\vec{CD} - \vec{AC} = 0$

Loop 2: $\vec{DE} + \vec{EG} -\vec{GF} - \vec{DF} = 0$

Loop 3: $\vec{GA} + \vec{HJ} -\vec{IJ} - \vec{IG} = 0$

Loop 4: $\vec{HK} -\vec{JK} - \vec{HJ} = 0$

**n x-y-coördinaten geeft dit:**

Loop 1:

x: $L_2 cos(\theta_2) + L_{3a} cos(\theta_3) - L_{4a} cos(\theta_4) - L_{1} cos(\theta_1) = 0$

y: $L_2 sin(\theta_2) + L_{3a} sin(\theta_3) - L_{4a} sin(\theta_4) - L_{1} sin(\theta_1) = 0$

Loop 2:

x: $L_{4b} cos(\theta_4) + L_{5a} cos(\theta_5) - L_{6a} cos(\theta_6) - L_{3b} cos(\theta_3) = 0$

y: $L_{4b} sin(\theta_4) + L_{5a} sin(\theta_5) - L_{6a} sin(\theta_6) - L_{3b} sin(\theta_3) = 0$

Loop 3:

x: $L_{6b} cos(\theta_6) + L_{8} cos(\theta_8) - L_{7} cos(\theta_7) - L_{5b} cos(\theta_5) = 0$

y: $L_{6b} sin(\theta_6) + L_{8} sin(\theta_8) - L_{7} sin(\theta_7) - L_{5b} sin(\theta_5) = 0$

Loop 4:

x: $L_{10} cos(\theta_{10}) - L_{9} cos(\theta_9) - L_{8} cos(\theta_8) = 0$

y: $L_{10} sin(\theta_{10}) - L_{9} sin(\theta_9) - L_{8} sin(\theta_8) = 0$

We nemen aan dat we de geometrie van het mechanisme kennen, hierdoor kennen we de lengte van de stangen $L_1, L_2, L_{3a}, L_{3b}, L_{4a}, L_{4b}, L_{5a}, L_{6a}, L_{6b}, L_7, L_8, L_9, L_{10}$ en de hoeken $\theta_1, \theta_9, \theta_{10}$.
De hoek $\theta_2$ wordt door de actuator aangedreven, waardoor de waarde hiervan op elk moment gekend is. We hebben acht vergelijkingen en 6 onbekenden $\theta_3, \theta_4, \theta_5, \theta_6, \theta_7, \theta_8$; dit kan door de numerical solver opgelost worden.

In [253]:
# Herman
# Animation of four bar

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
import matplotlib.animation as animation

%matplotlib widget

In [254]:
# Example analysis of a four-bar mechanism.

# Data initialization (all data is in SI units)

# kinematic parameters (link lengths)
L1 = 23 #mm
L2 = 25 #mm
L3a= 22 #mm
L3b= 27 #mm
L4a= 26 #mm
L4b= 19 #mm
L5a= 32 #mm
L5b= 23 #mm
L6a= 14 #mm
L6b= 21 #mm
L7= 20 #mm
L8= 23 #mm
L9= 31 #mm
L10= 54 #mm

#hoeken worden altijd gemeten vanaf de horizontale (tegenwijzerszin)
theta_1 = 15 * np.pi / 180  #rad
theta_9 =  ... #rad
theta_10 = ... #rad

In [ ]:
# STEP 1. Determination of Kinematics

# position analysis
theta_3_init = 18*np.pi/180  # initial condition for first step of position analysis with fsolve (phi3 and phi4)
theta_4_init = 96*np.pi/180  # VERY IMPORTANT because it determines which branch of the mechanism you're in
theta_5_init = 5*np.pi/180
theta_6_init = 72*np.pi/180
theta_7_init = 71*np.pi/180
theta_8_init = 2*np.pi/180
print(theta_3_init, theta_4_init, theta_5_init, theta_6_init, theta_7_init, theta_8_init)

t_begin = 0  # start time of simulation
t_end = 20  # end time of simulation
Ts = 0.05  # time step of simulation
t = np.arange(t_begin, t_end + Ts, Ts)  # time vector

# initialization of driver #afgeleides van phi2
omega = 0.5
A = 93*np.pi/180 
theta_2 = 93*np.pi/180 + A * np.sin(omega * t)
dtheta_2 = omega * A * np.cos(omega * t)
ddtheta_2 = -omega ** 2 * A * np.sin(omega * t)

0.3141592653589793 1.6755160819145563 0.08726646259971647 1.2566370614359172 1.239183768915974 0.03490658503988659


In [256]:
# Define rotation function
def rotate_vector(z, theta):
    rotation_matrix = np.array([[np.cos(theta), -np.sin(theta)],
                                [np.sin(theta), np.cos(theta)]])
    return np.dot(rotation_matrix, z)

In [257]:
# Define loop closure equations
def loop_closure_eqs(theta_init, theta_2, L1, L2, L3a, L3b, L4a, L4b, L5a, L5b, L6a, L6b, L7, L8, L9, L10, theta_1, theta_9, theta_10):
    theta_3 = theta_init[0]
    theta_4 = theta_init[1]
    theta_5 = theta_init[2]
    theta_6 = theta_init[3]
    theta_7 = theta_init[4] 
    theta_8 = theta_init[5]

    # Loop closure equations
    F1 = L2 * np.cos(theta_2) + L3a * np.cos(theta_3) - L4a * np.cos(theta_4) - L1 * np.cos(theta_1) #+ ipv - voor L4a
    F2 = L2 * np.sin(theta_2) + L3a * np.sin(theta_3) - L4a * np.sin(theta_4) - L1 * np.sin(theta_1)
    F3 = L4b * np.cos(theta_4) + L5a * np.cos(theta_5) - L6a * np.cos(theta_6) - L3b * np.cos(theta_3)
    F4 = L4b * np.sin(theta_4) + L5a * np.sin(theta_5) - L6a * np.sin(theta_6) - L3b * np.sin(theta_3)
    F5 = L6b * np.cos(theta_6) + L8 * np.cos(theta_8) - L7 * np.cos(theta_7) - L5b * np.cos(theta_5)
    F6 = L6b * np.sin(theta_6) + L8 * np.sin(theta_8) - L7 * np.sin(theta_7) - L5b * np.sin(theta_5)
   
    return [F1, F2, F3, F4, F5, F6]

In [260]:
# Define kinematic computations of the four-bar mechanism

def kinematics_8bar(theta_init, L1, L2, L3a, L3b, L4a, L4b, L5a, L5b, L6a, L6b, L7, L8, L9, L10, theta_1, theta_2, theta_9, theta_10, dtheta_2, ddtheta_2, theta_3_init, theta_4_init, theta_5_init, theta_6_init, theta_7_init, theta_8_init, t, fig_kin_8bar): #gekende hoeken gaan erin, onbekende hoeken worden init
    theta_3 = np.zeros_like(t);                           dtheta_3 = np.zeros_like(t);                ddtheta_3 = np.zeros_like(t)
    theta_4 = np.zeros_like(t);                           dtheta_4 = np.zeros_like(t);                ddtheta_4 = np.zeros_like(t)
    theta_5 = np.zeros_like(t);                           dtheta_5 = np.zeros_like(t);                ddtheta_5 = np.zeros_like(t)
    theta_6 = np.zeros_like(t);                           dtheta_6 = np.zeros_like(t);                ddtheta_6 = np.zeros_like(t)
    theta_7 = np.zeros_like(t);                           dtheta_7 = np.zeros_like(t);                ddtheta_7 = np.zeros_like(t)
    theta_8 = np.zeros_like(t);                           dtheta_8 = np.zeros_like(t);                ddtheta_8 = np.zeros_like(t)
    
    optim_options = {"full_output":True}  # options for fsolve

    for k, time in enumerate(t):
        # Position Analysis
        x, _, ier, message  = fsolve(lambda x: loop_closure_eqs(x, theta_2[k], L1, L2, L3a, L3b, L4a, L4b, L5a, L5b, L6a, L6b, L7, L8, L9, L10, theta_1, theta_9, theta_10), [theta_3_init, theta_4_init, theta_5_init, theta_6_init, theta_7_init, theta_8_init], **optim_options)

        #if ier != 1:
         #   print("The fsolve exit flag was not 1, probably no convergence!")
         #   print(message)

        theta_3[k] = x[0]
        theta_4[k] = x[1]
        theta_5[k] = x[2]
        theta_6[k] = x[3]
        theta_7[k] = x[4]
        theta_8[k] = x[5]
        print("resultaten hoeken in graden: ", np.degrees(theta_3[k]), np.degrees(theta_4[k]), np.degrees(theta_5[k]), np.degrees(theta_6[k]), np.degrees(theta_7[k]), np.degrees(theta_8[k]))

#kinematics_8bar(theta_init=[theta_3_init, theta_4_init, theta_5_init, theta_6_init, theta_7_init, theta_8_init], L1=L1, L2=L2, L3a=L3a, L3b=L3b, L4a=L4a, L4b=L4b, L5a=L5a, L5b=L5b, L6a=L6a, L6b=L6b, L7=L7, L8=L8, L9=L9, L10=L10, theta_1=theta_1, theta_2=theta_2, theta_9=theta_9, theta_10=theta_10, dtheta_2=dtheta_2, ddtheta_2=ddtheta_2, theta_3_init=theta_3_init, theta_4_init=theta_4_init, theta_5_init=theta_5_init, theta_6_init=theta_6_init, theta_7_init=theta_7_init, theta_8_init=theta_8_init, t=t, fig_kin_8bar=None)
#kinematics_8bar(theta_init=[theta_3_init, theta_4_init, theta_5_init, theta_6_init, theta_7_init, theta_8_init], L1=L1, L2=L2, L3a=L3a, L3b=L3b, L4a=L4a, L4b=L4b, L5a=L5a, L5b=L5b, L6a=L6a, L6b=L6b, L7=L7, L8=L8, L9=L9, L10=L10, theta_1=theta_1, theta_2=theta_2, theta_9=theta_9, theta_10=theta_10, dtheta_2=dtheta_2, ddtheta_2=ddtheta_2, theta_3_init=theta_3_init, theta_4_init=theta_4_init, theta_5_init=theta_5_init, theta_6_init=theta_6_init, theta_7_init=theta_7_init, theta_8_init=theta_8_init, t=t, fig_kin_8bar=None)
        # Velocity Analysis
        A = np.array([[-L3a * np.sin(theta_3[k]), L4a * np.sin(theta_4[k]), 0, 0, 0, 0],
                [L3a * np.cos(theta_3[k]), -L4a * np.cos(theta_4[k]), 0, 0, 0, 0],
                [L3b*np.sin(theta_3[k]), -L4b*np.sin(theta_4[k]), -L5a*np.sin(theta_5[k]), L6a*np.sin(theta_6[k]), 0, 0],
                [-L3b*np.cos(theta_3[k]), L4b*np.cos(theta_4[k]), L5a*np.cos(theta_5[k]), -L6a*np.cos(theta_6[k]), 0, 0],
                [0, 0, L5b*np.sin(theta_5[k]), -L6b*np.sin(theta_6[k]), L7*np.sin(theta_7[k]), -L8*np.sin(theta_8[k])],
                [0, 0, -L5b*np.cos(theta_5[k]), L6b*np.cos(theta_6[k]), -L7*np.cos(theta_7[k]), L8*np.cos(theta_8[k])]])
        B = np.array([L2 * np.sin(theta_2[k]) * dtheta_2[k],
                -L2 * np.cos(theta_2[k]) * dtheta_2[k], [0], [0], [0], [0]])

x = np.linalg.solve(A, B)
dtheta_3[k] = x[0]
dtheta_4[k] = x[1]
dtheta_5[k] = x[2]
dtheta_6[k] = x[3]
dtheta_7[k] = x[4]
dtheta_8[k] = x[5]


# Acceleration Analysis
B = np.array([L2 * np.cos(theta_2[k]) * dtheta_2[k]**2 + L2 * np.sin(theta_2[k]) * ddtheta_2[k] + L3a * np.cos(theta_3[k]) * dtheta_3[k]**2 - L4a * np.cos(theta_4[k]) * dtheta_4[k]**2,
                L2 * np.sin(theta_2[k]) * dtheta_2[k]**2 - L2 * np.cos(theta_2[k]) * ddtheta_2[k] + L3a * np.sin(theta_3[k]) * dtheta_3[k]**2 - L4a * np.sin(theta_4[k]) * dtheta_4[k]**2],
                [L4b*np.cos(theta_4[k]) * dtheta_4[k]**2 + L5a * np.cos(theta_5[k]) * dtheta_5[k]**2 - L6a * np.cos(theta_6[k]) * dtheta_6[k]**2 - L3b * np.cos(theta_3[k]) * dtheta_3[k]**2],
                [L4b*np.sin(theta_4[k]) * dtheta_4[k]**2 + L5a * np.sin(theta_5[k]) * dtheta_5[k]**2 - L6a * np.sin(theta_6[k]) * dtheta_6[k]**2 - L3b * np.sin(theta_3[k]) * dtheta_3[k]**2],
                [L6b*np.cos(theta_6[k]) * dtheta_6[k]**2 + L8 * np.cos(theta_8[k]) * dtheta_8[k]**2 - L7 * np.cos(theta_7[k]) * dtheta_7[k]**2 - L5b * np.cos(theta_5[k]) * dtheta_5[k]**2],
                [L6b*np.sin(theta_6[k]) * dtheta_6[k]**2 + L8 * np.sin(theta_8[k]) * dtheta_8[k]**2 - L7 * np.sin(theta_7[k]) * dtheta_7[k]**2 - L5b * np.sin(theta_5[k]) * dtheta_5[k]**2])
x = np.linalg.solve(A, B)
ddtheta_3[k] = x[0]
ddtheta_4[k] = x[1]
ddtheta_5[k] = x[2]
ddtheta_6[k] = x[3]
ddtheta_7[k] = x[4]
ddtheta_8[k] = x[5]



# Next iteration step
theta3_init = theta3[k] + (t[1] - t[0]) * dtheta3[k]
theta4_init = theta4[k] + (t[1] - t[0]) * dtheta4[k]
theta5_init = theta5[k] + (t[1] - t[0]) * dtheta5[k]
theta6_init = theta6[k] + (t[1] - t[0]) * dtheta6[k]
theta7_init = theta7[k] + (t[1] - t[0]) * dtheta7[k]
theta8_init = theta8[k] + (t[1] - t[0]) * dtheta8[k]

######## Animation #########

t_size = len(t)
# Define point P
# P = 0
P = np.array([0,0])
# Define point S

# S = r1 * np.exp(1j * phi1)
S = rotate_vector(np.array([r1, 0]), phi1)

# Define the number of frames in the movie
frames = 200
# Calculate the time between frames
delta = int(np.floor(t_size / frames))
# Define the index vector for frames
index_vec = np.arange(0, t_size, delta)

x_left = -1.5 * r2
y_bottom = -1.5 * max(r2, r4)
x_right = r1 + 1.5 * r4
y_top = 1.5 * max(r2, r4)
# Define the function to update the plot for each frame
movie_axes = [x_left, x_right, y_bottom, y_top]
def update(frame):
    plt.cla()  # Clear the current axes
    plt.axis('equal')  # Ensures 1 unit on x == 1 unit on y

    # Calculate Cartesian coordinates using rotation matrix
    Q = P + rotate_vector(np.array([r2, 0]), phi2[frame])
    R1 = Q + rotate_vector(np.array([r3, 0]), phi3[frame])
    R2 = S + rotate_vector(np.array([r4, 0]), phi4[frame])

    loop1 = np.array([P, Q, R1, R2, S])

    # plt.plot(loop1.real, loop1.imag, '-o')

    plt.plot(loop1[:, 0], loop1[:, 1], '-o')
    plt.axis(movie_axes)
    plt.title(f'Frame {frame}')
    plt.xlabel('[m]')
    plt.ylabel('[m]')

# Initialize the figure
fig = plt.figure()
ax = fig.add_subplot(111)
ax.set_xlim((x_left, x_right))
ax.set_ylim((y_bottom, y_top))

# Create the animation
global ani 
ani = animation.FuncAnimation(fig, update, frames=len(index_vec), interval=50)
plt.show()



NameError: name 'B' is not defined

# Snelheidsanalyse gebaseerd op de sluitingsvergelijkingen
-> tijdsafgeleide van de positievergelijkingen


Loop 1:

x: $- L_2 sin(\theta_2)\omega_2 - L_{3a} sin(\theta_3)\omega_3 + L_{4a} sin(\theta_4)\omega_4  = 0$

y: $L_2 cos(\theta_2)\omega_2 + L_{3a} cos(\theta_3)\omega_3 - L_{4a} cos(\theta_4)\omega_4  = 0$

Loop 2:

x: $ - L_{4b} sin(\theta_4)\omega_4 - L_{5a} sin(\theta_5) \omega_5 + L_{6a} sin(\theta_6) \omega_6 + L_{3b} sin(\theta_3) \omega_3 = 0$

y: $L_{4b} cos(\theta_4) \omega_4 + L_{5a} cos(\theta_5) \omega_5 - L_{6a} cos(\theta_6) \omega_6 - L_{3b} cos(\theta_3) \omega_3 = 0$

Loop 3:

x: $- L_{6b} sin(\theta_6) \omega_6 - L_{8} sin(\theta_8) \omega_8 + L_{7} sin(\theta_7) \omega_7 + L_{5b} sin(\theta_5) \omega_5 = 0$

y: $L_{6b} cos(\theta_6) \omega_6 + L_{8} cos(\theta_8) \omega_8 - L_{7} cos(\theta_7) \omega_7 - L_{5b} cos(\theta_5) \omega_5 = 0$

Loop 4:

x: $- L_{10} sin(\theta_{10}) \omega_{10} + L_{9} sin(\theta_9) \omega_9 + L_{8} sin(\theta_8) \omega_8 = 0$

y: $L_{10} cos(\theta_{10}) \omega_{10} - L_{9} cos(\theta_9) \omega_9 - L_{8} cos(\theta_8) \omega_8 = 0$

## Versnellingsanalyse gebaseerd op de sluitingsvergelijkingen

Loop 1:

x: $- L_2 cos(\theta_2)\omega_2^2 - L_{3a} cos(\theta_3)\omega_3^2 + L_{4a} cos(\theta_4)\omega_4^2 - L_2 sin(\theta_2)\alpha_2 - L_{3a} sin(\theta_3)\alpha_3 + L_{4a} sin(\theta_4)\alpha_4  = 0$

y: $- L_2 sin(\theta_2)\omega_2^2 - L_{3a} sin(\theta_3)\omega_3^2 + L_{4a} sin(\theta_4)\omega_4^2 + L_2 cos(\theta_2)\alpha_2 + L_{3a} cos(\theta_3)\alpha_3 - L_{4a} cos(\theta_4)\alpha_4 = 0$

Loop 2:

x: $ - L_{4b} cos(\theta_4)\omega_4^2 - L_{5a} cos(\theta_5) \omega_5^2 + L_{6a} cos(\theta_6) \omega_6^2 + L_{3b} cos(\theta_3) \omega_3^2 - L_{4b} sin(\theta_4)\alpha_4 - L_{5a} sin(\theta_5) \alpha_5 + L_{6a} sin(\theta_6) \alpha_6 + L_{3b} sin(\theta_3) \alpha_3= 0$

y: $- L_{4b} sin(\theta_4) \omega_4^2 - L_{5a} sin(\theta_5) \omega_5^2 + L_{6a} sin(\theta_6) \omega_6^2 + L_{3b} sin(\theta_3) \omega_3^2 + L_{4b} cos(\theta_4) \alpha_4 + L_{5a} cos(\theta_5) \alpha_5 - L_{6a} cos(\theta_6) \alpha_6 - L_{3b} cos(\theta_3) \alpha_3 = 0$
Loop 3:

x: $- L_{6b} cos(\theta_6) \omega_6^2 - L_{8} cos(\theta_8) \omega_8^2 + L_{7} cos(\theta_7) \omega_7^2 + L_{5b} cos(\theta_5) \omega_5^2 - L_{6b} sin(\theta_6) \alpha_6 - L_{8} sin(\theta_8) \alpha_8 + L_{7} sin(\theta_7) \alpha_7 + L_{5b} sin(\theta_5) \alpha_5= 0$

y: $- L_{6b} sin(\theta_6) \omega_6^2 - L_{8} sin(\theta_8) \omega_8^2 + L_{7} sin(\theta_7) \omega_7^2 + L_{5b} sin(\theta_5) \omega_5^2 + _{6b} cos(\theta_6) \alpha_6 + L_{8} cos(\theta_8) \alpha_8 - L_{7} cos(\theta_7) \alpha_7 - L_{5b} cos(\theta_5) \alpha_5 = 0$

In [259]:
# Velocity Analysis
A = np.array([[-L3a * np.sin(theta_3[k]), L4a * np.sin(theta_4[k]), 0, 0, 0, 0],
                [L3a * np.cos(theta_3[k]), -L4a * np.cos(theta_4[k]), 0, 0, 0, 0],
                [L3b*np.sin(theta_3[k]), -L4b*np.sin(theta_4[k]), -L5a*np.sin(theta_5[k]), L6a*np.sin(theta_6[k]), 0, 0],
                [-L3b*np.cos(theta_3[k]), L4b*np.cos(theta_4[k]), L5a*np.cos(theta_5[k]), -L6a*np.cos(theta_6[k]), 0, 0],
                [0, 0, L5b*np.sin(theta_5[k]), -L6b*np.sin(theta_6[k]), L7*np.sin(theta_7[k]), -L8*np.sin(theta_8[k])],
                [0, 0, -L5b*np.cos(theta_5[k]), L6b*np.cos(theta_6[k]), -L7*np.cos(theta_7[k]), L8*np.cos(theta_8[k])]])
B = np.array([L2 * np.sin(theta_2[k]) * dtheta_2[k],
                -L2 * np.cos(theta_2[k]) * dtheta_2[k], [0], [0], [0], [0]])

x = np.linalg.solve(A, B)
dtheta_3[k] = x[0]
dtheta_4[k] = x[1]
dtheta_5[k] = x[2]
dtheta_6[k] = x[3]
dtheta_7[k] = x[4]
dtheta_8[k] = x[5]


# Acceleration Analysis
B = np.array([L2 * np.cos(theta_2[k]) * dtheta_2[k]**2 + L2 * np.sin(theta_2[k]) * ddtheta_2[k] + L3a * np.cos(theta_3[k]) * dtheta_3[k]**2 - L4a * np.cos(theta_4[k]) * dtheta_4[k]**2,
                L2 * np.sin(theta_2[k]) * dtheta_2[k]**2 - L2 * np.cos(theta_2[k]) * ddtheta_2[k] + L3a * np.sin(theta_3[k]) * dtheta_3[k]**2 - L4a * np.sin(theta_4[k]) * dtheta_4[k]**2],
                [L4b*np.cos(theta_4[k]) * dtheta_4[k]**2 + L5a * np.cos(theta_5[k]) * dtheta_5[k]**2 - L6a * np.cos(theta_6[k]) * dtheta_6[k]**2 - L3b * np.cos(theta_3[k]) * dtheta_3[k]**2],
                [L4b*np.sin(theta_4[k]) * dtheta_4[k]**2 + L5a * np.sin(theta_5[k]) * dtheta_5[k]**2 - L6a * np.sin(theta_6[k]) * dtheta_6[k]**2 - L3b * np.sin(theta_3[k]) * dtheta_3[k]**2],
                [L6b*np.cos(theta_6[k]) * dtheta_6[k]**2 + L8 * np.cos(theta_8[k]) * dtheta_8[k]**2 - L7 * np.cos(theta_7[k]) * dtheta_7[k]**2 - L5b * np.cos(theta_5[k]) * dtheta_5[k]**2],
                [L6b*np.sin(theta_6[k]) * dtheta_6[k]**2 + L8 * np.sin(theta_8[k]) * dtheta_8[k]**2 - L7 * np.sin(theta_7[k]) * dtheta_7[k]**2 - L5b * np.sin(theta_5[k]) * dtheta_5[k]**2])
x = np.linalg.solve(A, B)
ddtheta_3[k] = x[0]
ddtheta_4[k] = x[1]
ddtheta_5[k] = x[2]
ddtheta_6[k] = x[3]
ddtheta_7[k] = x[4]
ddtheta_8[k] = x[5]



# Next iteration step
theta3_init = theta3[k] + (t[1] - t[0]) * dtheta3[k]
theta4_init = theta4[k] + (t[1] - t[0]) * dtheta4[k]
theta5_init = theta5[k] + (t[1] - t[0]) * dtheta5[k]
theta6_init = theta6[k] + (t[1] - t[0]) * dtheta6[k]
theta7_init = theta7[k] + (t[1] - t[0]) * dtheta7[k]
theta8_init = theta8[k] + (t[1] - t[0]) * dtheta8[k]

######## Animation #########

t_size = len(t)
# Define point P
# P = 0
P = np.array([0,0])
# Define point S

# S = r1 * np.exp(1j * phi1)
S = rotate_vector(np.array([r1, 0]), phi1)

# Define the number of frames in the movie
frames = 200
# Calculate the time between frames
delta = int(np.floor(t_size / frames))
# Define the index vector for frames
index_vec = np.arange(0, t_size, delta)

x_left = -1.5 * r2
y_bottom = -1.5 * max(r2, r4)
x_right = r1 + 1.5 * r4
y_top = 1.5 * max(r2, r4)
# Define the function to update the plot for each frame
movie_axes = [x_left, x_right, y_bottom, y_top]
def update(frame):
    plt.cla()  # Clear the current axes
    plt.axis('equal')  # Ensures 1 unit on x == 1 unit on y

    # Calculate Cartesian coordinates using rotation matrix
    Q = P + rotate_vector(np.array([r2, 0]), phi2[frame])
    R1 = Q + rotate_vector(np.array([r3, 0]), phi3[frame])
    R2 = S + rotate_vector(np.array([r4, 0]), phi4[frame])

    loop1 = np.array([P, Q, R1, R2, S])

    # plt.plot(loop1.real, loop1.imag, '-o')

    plt.plot(loop1[:, 0], loop1[:, 1], '-o')
    plt.axis(movie_axes)
    plt.title(f'Frame {frame}')
    plt.xlabel('[m]')
    plt.ylabel('[m]')

# Initialize the figure
fig = plt.figure()
ax = fig.add_subplot(111)
ax.set_xlim((x_left, x_right))
ax.set_ylim((y_bottom, y_top))

# Create the animation
global ani 
ani = animation.FuncAnimation(fig, update, frames=len(index_vec), interval=50)
plt.show()

NameError: name 'k' is not defined